정수계획 모형

In [2]:
from ortools.linear_solver import pywraplp

# Create the mip solver with the SCIP backend.
solver = pywraplp.Solver.CreateSolver("SAT")

infinity = solver.infinity()
# x and y are integer non-negative variables.
# x = solver.IntVar(0.0,infinity, "x")
# y = solver.IntVar(0.0,infinity, "y")
x = solver.NumVar(0.0,infinity, "x")
y = solver.NumVar(0.0,infinity, "y")

solver.Add(x + 7*y <=17.5)
solver.Add(x<=3.5)

solver.Maximize(x+10*y)

print(f"Solving with {solver.SolverVersion()}")
status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print("Solution:")
    print("Objective value=", solver.Objective().Value())
    print("x=", x.solution_value())
    print("y=", y.solution_value())
else:
    print("The problem does not have an optimal solution.")


Solving with CP-SAT solver v9.15.6755
Solution:
Objective value= 25.0
x= 0.0
y= 2.5


예제1. 스태프 결정 문제 - Code

In [ ]:
from ortools.linear_solver import pywraplp

# Create the mip solver with the SCIP backend.
solver = pywraplp.Solver.CreateSolver("SAT")

REQ = [20,16,13,16,19,14,12]
names = ['MON', 'TUE', 'WED', 'THU', 'FRI', 'SAT', 'SUN']

infinity = solver.infinity()

x = {}
for i in range(7):
    x[i] = solver.IntVar(0,infinity, "x[%i]" %i)

for i in range(7):
    constraint_expr = [x[(i-j)%7] for j in range(5)]
    solver.Add(sum(constraint_expr)>=REQ[i], names[i])

objective = solver.Objective()
solver.Minimize(solver.Sum(x[i] for i in range(7)))

# 모델 파일 생성
with open('or4-2.lp', "w") as out_f:
    lp_text = solver.ExportModelAsLpFormat(False)
    out_f.write(lp_text)

status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print("Objective value=", solver.Objective().Value())
    for i in range(7):
        print(x[i].name(), "=", x[i].solution_value())
else:
    print("The problem does not have an optimal solution.")


예제2. 팀배정 문제

In [5]:
from ortools.linear_solver import pywraplp

# Create the mip solver with the SCIP backend.
solver = pywraplp.Solver.CreateSolver("SAT")

Ratings = [
    [None, 9, 3, 4, 2, 1, 5, 6],
    [None, None, 1, 7, 3, 5, 2, 1],
    [None, None, None, 4, 4, 2, 9, 2],
    [None, None, None, None, 1, 5, 5, 2],
    [None, None, None, None, None, 8, 7, 6],
    [None, None, None, None, None, None, 2, 3],
    [None, None, None, None, None, None, None, 4]
]

nC = len(Ratings[0])

# 의사결정 변수
X = {}
for i in range(nC):
    for j in range(nC):
        if i !=j:
            X[i,j] = solver.IntVar(0,1,"X"+str(i)+str(j))

# 제약조건
const_expr = []

# 컨설턴트당 1명만 배정
for i in range(nC):
    const_expr = [X[i,j] for j in range(nC) if i != j]
    solver.Add(sum(const_expr) == 1, 'consultant_' + str(i))

# xij = xji 제약
for i in range(nC):
    for j in range(nC):
        if i < j:
            solver.Add(X[i, j] == X[j, i], 'x_' + str(i) + '_' + str(j))

# 목적함수
obj_expr = []

for i in range(nC-1):
    for j in range(nC):
        if Ratings[i][j] != None:
            obj_expr.append(Ratings[i][j] * X[i, j])

solver.Maximize(solver.Sum(obj_expr))

# 모델 파일 생성
with open("or4-4.lp", "w") as out_f:
    lp_text = solver.ExportModelAsLpFormat(False)
    out_f.write(lp_text)

status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print("Objective value = %.1f" % solver.Objective().Value())
    for i in range(nC):
        for j in range(nC):
            if i != j and X[i, j].solution_value() != 0:
                print("Consultant [%i]" %i, " --> Consultant [%i]" %j)
else:
    print("The problem does not have an optimal solution.")

Objective value = 30.0
Consultant [0]  --> Consultant [7]
Consultant [1]  --> Consultant [3]
Consultant [2]  --> Consultant [6]
Consultant [3]  --> Consultant [1]
Consultant [4]  --> Consultant [5]
Consultant [5]  --> Consultant [4]
Consultant [6]  --> Consultant [2]
Consultant [7]  --> Consultant [0]


예제3. 정수 계획 문제

In [6]:
from ortools.linear_solver import pywraplp

solver = pywraplp.Solver.CreateSolver("SCIP")

nVars = 4
values = [9, 5, 6, 4]
reqs = [6, 3, 5, 2]

# 의사결정변수
x = {}
for i in range(nVars):
    x[i] = solver.IntVar(0, 1, "x[%i]" % i)

# 제약조건
solver.Add(sum([reqs[i]*x[i] for i in range(nVars)]) <= 11, 'const 0')
solver.Add(x[2] + x[3] <= 1, 'const 1')
solver.Add(x[2] <= x[0], 'const 2')
solver.Add(x[3] <= x[1], 'const 3')

# 목적함수
solver.Maximize(sum([values[i]*x[i] for i in range(nVars)]))

with open('or5-1.lp', "w") as out_f:
    lp_text = solver.ExportModelAsLpFormat(False)
    out_f.write(lp_text)

status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print("OPTIMAL")
    print("목적함수값 = ", solver.Objective().Value())
    for i in range(nVars):
        print(x[i].name(), " = ", x[i].solution_value())
else:
    print("The problem does not have an optimal solution.")

OPTIMAL
목적함수값 =  18.0
x[0]  =  1.0
x[1]  =  1.0
x[2]  =  0.0
x[3]  =  1.0


예제4_교수님이 문제가 이상하다고 인정해주심

In [7]:
from ortools.linear_solver import pywraplp

solver = pywraplp.Solver.CreateSolver("SCIP")
infinity = solver.infinity()
BIGM = 1000000

# 의사결정변수
x = {}
for i in range(2):
    x[i] = solver.IntVar(0, infinity, "x[%i]" % i)

y = {}
for i in range(3):
    y[i] = solver.IntVar(0, 1, "y[%i]" % i)

# 제약조건
solver.Add(x[0]/50 + x[1]/40 <= 500 + BIGM * y[0], 'const 0')
solver.Add(x[0]/40 + x[1]/25 <= 700 + BIGM * (1 - y[0]), 'const 1')
solver.Add(x[0] <= BIGM * y[1], 'const 2')
solver.Add(x[1] <= BIGM * y[2], 'const 3')

# 목적함수
solver.Maximize(10 * x[0] + 15 * x[1] - 50000 * y[1] - 80000 * y[2])

with open('or5-1.lp', "w") as out_f:
    lp_text = solver.ExportModelAsLpFormat(False)
    out_f.write(lp_text)

status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print("OPTIMAL")
    print("목적함수값 = ", solver.Objective().Value())
    for i in range(2):
        print(x[i].name(), " = ", x[i].solution_value())
    for i in range(3):
        print(y[i].name(), " = ", y[i].solution_value())
else:
    print("The problem does not have an optimal solution.")

OPTIMAL
목적함수값 =  230000.0
x[0]  =  28000.0
x[1]  =  0.0
y[0]  =  1.0
y[1]  =  1.0
y[2]  =  0.0


실습1. 스태프 결정 문제 2 - Code

In [21]:
from ortools.linear_solver import pywraplp

solver = pywraplp.Solver.CreateSolver("SAT")

REQ = [2,2,2,2,2,2,8,8,8,8,
       4,4,3,3,3,3,6,6,5,5,
       5,5,3,3]

infinity = solver.infinity()

# 의사결정변수: i시에 일을 시작하는 근무자 수
x = {}
for i in range(24):
    x[i] = solver.IntVar(0, infinity, "x[%i]" % i)

# 제약조건: 각 시간대 수요 충족
for t in range(24):
    constraint_expr = [
        x[t],
        x[(t-1) % 24],
        x[(t-2) % 24],
        x[(t-3) % 24],
        x[(t-5) % 24],
        x[(t-6) % 24],
        x[(t-7) % 24],
        x[(t-8) % 24]
    ]
    solver.Add(sum(constraint_expr) >= REQ[t], "cover[%i]" % t)

# 목적함수: 총 고용 인원 최소화
solver.Minimize(solver.Sum(x[i] for i in range(24)))

# 모델 파일 생성
with open('or5-2-1.lp', "w") as out_f:
    lp_text = solver.ExportModelAsLpFormat(False)
    out_f.write(lp_text)

status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print("Objective value =", solver.Objective().Value())
    for i in range(24):
        print(x[i].name(), "=", x[i].solution_value())
else:
    print("The problem does not have an optimal solution.")

Objective value = 16.0
x[0] = 0.0
x[1] = 1.0
x[2] = 0.0
x[3] = 1.0
x[4] = 1.0
x[5] = 1.0
x[6] = 4.0
x[7] = 1.0
x[8] = 0.0
x[9] = 0.0
x[10] = 0.0
x[11] = 1.0
x[12] = 0.0
x[13] = 0.0
x[14] = 2.0
x[15] = 2.0
x[16] = 1.0
x[17] = 0.0
x[18] = 1.0
x[19] = 0.0
x[20] = 0.0
x[21] = 0.0
x[22] = 0.0
x[23] = 0.0


실습2. 컨설턴스 배정 최적화

In [ ]:
from ortools.linear_solver import pywraplp

solver = pywraplp.Solver.CreateSolver("SCIP")

Ratings = [
    [15, 18, 21, 12, 25, 19, 14, 22],
    [20, 14, 17, 23, 16, 21, 18, 13],
    [11, 22, 19, 16, 20, 15, 23, 17],
    [24, 16, 13, 20, 18, 22, 11, 19],
    [17, 20, 22, 14, 13, 17, 20, 16],
    [19, 13, 16, 21, 22, 14, 17, 20],
    [22, 17, 20, 18, 15, 20, 16, 21]
]

nConsultants = len(Ratings)       # 7
nProjects = len(Ratings[0])       # 8

# 의사결정변수
X = {}
for i in range(nConsultants):
    for j in range(nProjects):
        X[i, j] = solver.IntVar(0, 1, "X[%i,%i]" % (i, j))

# 제약조건 1: 각 컨설턴트는 정확히 1개의 프로젝트에 배정
for i in range(nConsultants):
    solver.Add(sum(X[i, j] for j in range(nProjects)) == 1, "consultant_%i" % i)

# 제약조건 2: 각 프로젝트는 최대 1명의 컨설턴트만 배정
for j in range(nProjects):
    solver.Add(sum(X[i, j] for i in range(nConsultants)) <= 1, "project_%i" % j)

# 목적함수: 총 배정 비용 최소화
obj_expr = []
for i in range(nConsultants):
    for j in range(nProjects):
        obj_expr.append(Ratings[i][j] * X[i, j])

solver.Minimize(solver.Sum(obj_expr))

# 모델 파일 생성
with open("or5-2-2.lp", "w") as out_f:
    lp_text = solver.ExportModelAsLpFormat(False)
    out_f.write(lp_text)

status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print("Objective value = %.1f" % solver.Objective().Value())
    for i in range(nConsultants):
        for j in range(nProjects):
            if X[i, j].solution_value() == 1:
                print("Consultant [%i] --> Project [%i]" % (i + 1, j + 1))
else:
    print("The problem does not have an optimal solution.")

Objective value = 91.0
Consultant [1] --> Project [4]
Consultant [2] --> Project [8]
Consultant [3] --> Project [1]
Consultant [4] --> Project [3]
Consultant [5] --> Project [5]
Consultant [6] --> Project [2]
Consultant [7] --> Project [7]


실습3

In [18]:
from ortools.linear_solver import pywraplp

solver = pywraplp.Solver.CreateSolver("SCIP")
infinity = solver.infinity()
BIGM = 100000000

# 의사결정변수
x = {}
for i in range(3):
    x[i] = solver.IntVar(0, infinity, "x[%i]" % i)

y = {}
for i in range(2):
    y[i] = solver.IntVar(0, 1, "y[%i]" % i)

# 제약조건
solver.Add(0.2 * x[0] + 0.4 * x[1] + 0.2 * x[2] <= 1, 'const 0')
solver.Add(x[0] <= BIGM * y[0], 'const 1')
solver.Add(x[1] <= BIGM * y[1], 'const 2')
solver.Add(x[0] <= 3, 'const 3')
solver.Add(x[1] <= 2, 'const 4')
solver.Add(x[2] <= 5, 'const 5')

# 목적함수
solver.Maximize(2 * x[0] + 3 * x[1] + 0.8 * x[2] - 3 * y[0] - 2 * y[1])

# 모델 파일 생성
with open('or_example.lp', "w") as out_f:
    lp_text = solver.ExportModelAsLpFormat(False)
    out_f.write(lp_text)

status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print("OPTIMAL")
    print("목적함수값 = ", solver.Objective().Value())
    for i in range(3):
        print(x[i].name(), " = ", x[i].solution_value())
    for i in range(2):
        print(y[i].name(), " = ", y[i].solution_value())
else:
    print("The problem does not have an optimal solution.")

OPTIMAL
목적함수값 =  4.800000000000001
x[0]  =  0.0
x[1]  =  2.0
x[2]  =  1.0
y[0]  =  0.0
y[1]  =  1.0


실습4

In [19]:
from ortools.linear_solver import pywraplp

solver = pywraplp.Solver.CreateSolver("SCIP")
infinity = solver.infinity()
BIGM = 100000000

# 의사결정변수
x = {}
for i in range(4):
    x[i] = solver.IntVar(0, infinity, "x[%i]" % i)

y = {}
for i in range(5):
    y[i] = solver.IntVar(0, 1, "y[%i]" % i)

# 제약조건
solver.Add(y[0] + y[1] + y[2] + y[3] <= 2, 'const 0')
solver.Add(y[2] <= y[0] + y[1], 'const 1')
solver.Add(y[3] <= y[0] + y[1], 'const 2')
solver.Add(5 * x[0] + 3 * x[1] + 6 * x[2] + 4 * x[3] <= 6000 + BIGM * y[4], 'const 3')
solver.Add(4 * x[0] + 6 * x[1] + 3 * x[2] + 5 * x[3] <= 6000 + BIGM * (1 - y[4]), 'const 4')
solver.Add(x[0] <= BIGM * y[0], 'const 5')
solver.Add(x[1] <= BIGM * y[1], 'const 6')
solver.Add(x[2] <= BIGM * y[2], 'const 7')
solver.Add(x[3] <= BIGM * y[3], 'const 8')

# 목적함수
solver.Maximize(
    70 * x[0] - 50000 * y[0]
    + 60 * x[1] - 40000 * y[1]
    + 90 * x[2] - 70000 * y[2]
    + 80 * x[3] - 60000 * y[3]
)

# 모델 파일 생성
with open('or_product_mix.lp', "w") as out_f:
    lp_text = solver.ExportModelAsLpFormat(False)
    out_f.write(lp_text)

status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print("OPTIMAL")
    print("목적함수값 = ", solver.Objective().Value())
    for i in range(4):
        print(x[i].name(), " = ", x[i].solution_value())
    for i in range(5):
        print(y[i].name(), " = ", y[i].solution_value())
else:
    print("The problem does not have an optimal solution.")

OPTIMAL
목적함수값 =  80000.0
x[0]  =  0.0
x[1]  =  2000.0
x[2]  =  0.0
x[3]  =  0.0
y[0]  =  0.0
y[1]  =  1.0
y[2]  =  0.0
y[3]  =  0.0
y[4]  =  0.0


실습5

In [20]:
from ortools.linear_solver import pywraplp

solver = pywraplp.Solver.CreateSolver("SCIP")

# 대응시간(분)
T = [
    [5, 12, 30, 20, 15],
    [20, 4, 15, 10, 25],
    [15, 20, 6, 15, 12],
    [25, 15, 25, 4, 10],
    [10, 25, 15, 12, 5]
]

# 1일 평균 화재 횟수
F = [2, 1, 3, 1, 3]

n = 5

# 의사결정변수
# x[i] = 구역 i에 소방서를 설치하면 1, 아니면 0
x = {}
for i in range(n):
    x[i] = solver.IntVar(0, 1, "x[%i]" % i)

# y[i,j] = 구역 j의 화재를 구역 i의 소방서가 담당하면 1, 아니면 0
y = {}
for i in range(n):
    for j in range(n):
        y[i, j] = solver.IntVar(0, 1, "y[%i][%i]" % (i, j))

# 제약조건
# 소방서는 정확히 2개 설치
solver.Add(sum(x[i] for i in range(n)) == 2, "two_station")

# 각 구역의 화재는 정확히 1개의 소방서에 의해 담당
for j in range(n):
    solver.Add(sum(y[i, j] for i in range(n)) == 1, "assign_st_%i" % j)

# 소방서가 설치된 구역만 담당 가능
for i in range(n):
    for j in range(n):
        solver.Add(y[i, j] <= x[i], "st배치보장_%i_%i" % (i, j))

# 목적함수: 평균 화재 횟수를 가중치로 둔 총 대응시간 최소화
obj_expr = []
for i in range(n):
    for j in range(n):
        obj_expr.append(F[j] * T[i][j] * y[i, j])

solver.Minimize(solver.Sum(obj_expr))

# 모델 파일 생성
with open("or_assignment_firestation.lp", "w") as out_f:
    lp_text = solver.ExportModelAsLpFormat(False)
    out_f.write(lp_text)

status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print("OPTIMAL")
    print("목적함수값 = ", solver.Objective().Value())

    print("\n[소방서 설치 구역]")
    for i in range(n):
        print(x[i].name(), "=", x[i].solution_value())

    print("\n[구역별 담당 소방서]")
    for j in range(n):
        for i in range(n):
            if y[i, j].solution_value() == 1:
                print("구역 [%i] --> 소방서 [%i]" % (j + 1, i + 1))
else:
    print("The problem does not have an optimal solution.")

OPTIMAL
목적함수값 =  85.0

[소방서 설치 구역]
x[0] = 0.0
x[1] = 0.0
x[2] = 1.0
x[3] = 0.0
x[4] = 1.0

[구역별 담당 소방서]
구역 [1] --> 소방서 [5]
구역 [2] --> 소방서 [3]
구역 [3] --> 소방서 [3]
구역 [4] --> 소방서 [5]
구역 [5] --> 소방서 [5]
